In [0]:
!pip install yfinance

In [0]:
# Databricks notebook source
# MAGIC %md
# MAGIC # Stock Prices Bronze Ingestion

# COMMAND ----------

import yfinance as yf
import pandas as pd
from datetime import datetime

# COMMAND ----------

storage_account_name = "stmarketinsights"

bronze_path = f"abfss://bronze@{storage_account_name}.dfs.core.windows.net/stock_prices/"
bronze_table = "stock_prices_bronze"

tickers = ["AAPL", "MSFT", "NVDA", "TSLA", "AMZN"]
period = "1y"
interval = "1d"

In [0]:
def fetch_stock_data(tickers, period, interval):
    all_data = []

    for ticker in tickers:
        ticker_df = yf.Ticker(ticker).history(
            period=period,
            interval=interval,
            auto_adjust=False
        )

        ticker_df = ticker_df.reset_index()
        ticker_df["ticker"] = ticker

        all_data.append(ticker_df)

    return pd.concat(all_data, ignore_index=True)

In [0]:
raw_df = fetch_stock_data(tickers, period, interval)

display(raw_df.head())
print(raw_df.shape)

In [0]:
spark_df = spark.createDataFrame(raw_df)

display(spark_df.limit(5))

In [0]:
spark_df = spark.createDataFrame(raw_df)

In [0]:
spark_df = spark_df.withColumnsRenamed({
    "Date": "trade_date",
    "Open": "open_price",
    "High": "high_price",
    "Low": "low_price",
    "Close": "close_price",
    "Adj Close": "adjusted_close_price",
    "Volume": "volume",
    "Stock Splits": "stock_splits",
    "Dividends": "dividends"
})

In [0]:
(
    spark_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("market_insights.bronze.stock_prices")
)

In [0]:
display(
    spark.sql("SELECT * FROM market_insights.bronze.stock_prices LIMIT 10")
)